In [1]:
import torch
from torch import nn

torch.manual_seed(42)

vocab = {
    'good': 0,
    'bad': 1,
    'great': 2,
    'awful': 3,
    'delicious': 4,
    'terrible': 5,
    'excellent': 6,
    'poor': 7,
    'movie': 8,
    'food': 9,
    'service': 10,
}

X_train = torch.tensor([
    [vocab['good'], vocab['movie']],
    [vocab['bad'], vocab['movie']],
    [vocab['great'], vocab['food']],
    [vocab['awful'], vocab['food']],
    [vocab['delicious'], vocab['service']],
    [vocab['terrible'], vocab['service']],
    [vocab['excellent'], vocab['movie']],
    [vocab['poor'], vocab['food']],
], dtype=torch.long)

y_train = torch.tensor([1, 0, 1, 0, 1, 0, 1, 0])

X_test = torch.tensor([
    [vocab['good'], vocab['food']],
    [vocab['bad'], vocab['service']],
], dtype=torch.long)

y_test = torch.tensor([1, 0])

In [ ]:
class TinyTextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)
        pooled = embedded.mean(dim=1)
        logits = self.classifier(pooled)
        return logits

model = TinyTextClassifier(
    vocab_size=len(vocab),
    embedding_dim=8,
    num_classes=2,
)

In [3]:
with torch.no_grad():
    embedded = model.embedding(X_train)
    pooled = embedded.mean(dim=1)
    logits = model.classifier(pooled)

    print('Embedding:', embedded.shape)
    print('Pooling:', pooled.shape)
    print('Logits:', logits.shape)

Embedding: torch.Size([8, 2, 8])
Pooling: torch.Size([8, 8])
Logits: torch.Size([8, 2])


In [4]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

initial_loss = loss_fn(model(X_train), y_train).item()

for epoch in range(101):
    model.train()

    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.6f}')

final_loss = loss_fn(model(X_train), y_train).item()
print('Initial loss:', initial_loss)
print('Final loss:', final_loss)

assert final_loss < initial_loss

Epoch 0, Loss: 0.668118
Epoch 20, Loss: 0.000010
Epoch 40, Loss: 0.000000
Epoch 60, Loss: 0.000000
Epoch 80, Loss: 0.000000
Epoch 100, Loss: 0.000000
Initial loss: 0.6681176424026489
Final loss: 1.4901154088420299e-07


In [5]:
model.eval()

with torch.inference_mode():
    train_pred = model(X_train).argmax(dim=1)
    test_pred = model(X_test).argmax(dim=1)

    print('Train predictions:', train_pred)
    print('Train labels:', y_train)
    print('Test predictions:', test_pred)
    print('Test labels:', y_test)

    train_accuracy = (train_pred == y_train).float().mean()
    test_accuracy = (test_pred == y_test).float().mean()

    print('Train accuracy:', train_accuracy.item())
    print('Test accuracy:', test_accuracy.item())

Train predictions: tensor([1, 0, 1, 0, 1, 0, 1, 0])
Train labels: tensor([1, 0, 1, 0, 1, 0, 1, 0])
Test predictions: tensor([1, 0])
Test labels: tensor([1, 0])
Train accuracy: 1.0
Test accuracy: 1.0
